In [1]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [2]:
# # downloaded project inputs
# input_data_path, build_files_path = preprocess.unpack_files(data_files = '/data2/hratch/human_me/data.zip', 
#                                         build_files = '/data2/hratch/human_me/build_files.zip', 
#                                         data_out = '/data2/hratch/human_me/raw')
# preprocess.create_environment(input_data_path, build_files_path, root_path = '/home/hratch/Projects/human_me/',
#                               processed_data_path = '/data2/hratch/human_me/processed/', 
#                               n_cores = 20)

In [3]:
from preprocess import correct_inputs 

full model

In [4]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/recon2_2.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')
# # # optional - only if you want to express non-machinery proteins
# # correct_inputs.check_non_machinery(nonmachinery_file = '/data2/hratch/human_me/input_files/non_machinery.txt')

# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = False, compress_mrna = False)

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'me_model.pickle', 'wb') as handle:
#     pickle.dump(me_model, handle)

toy model

In [4]:
# correct_inputs.correct_model(model_file = '/data2/hratch/human_me/input_files/toy_model.xml', 
#                  psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# correct_inputs.correct_psim(psim_file = '/data2/hratch/human_me/input_files/psim_recon2_2.csv')

# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                                 model_id = 'toy_me_model')

# import pickle
# lp_path = '/data2/hratch/human_me/test_lp/'
# with open(lp_path + 'toy_me_model.pickle', 'wb') as handle:
#     pickle.dump(toy_me_model, handle)



../scripts/preprocess/correct_inputs.py:91 UserWarning: OIVD1m contains redundant complexes according to GPR, editing GPR
../scripts/preprocess/correct_inputs.py:135 UserWarning: gpi_hs does not exist in model. Adding to compartment r via sink This allows gpi_hs to be in the model at no cost.
../scripts/preprocess/correct_inputs.py:135 UserWarning: udpacgal does not exist in model. Adding to compartment g via sink This allows udpacgal to be in the model at no cost.
../scripts/preprocess/correct_inputs.py:159 UserWarning: Your metabolic model contains genes with HGNC:HGNC:####, changing to HGNC:####


Check for the recon2.2 HGNC:HGNC error
Remove genes not participating in reactions


# Check

In [5]:
from expression import build_me_model
toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                unmodeled_protein_frac = None,
                                                model_id = 'toy_me_model')
jabba = True
if jabba:
    for r in toy_me_model.reactions:
        if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
            r._lower_bound = -1000
            r._upper_bound = 1000
            
sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


Generate ubiquitin reactions for proteasomal degrdation
Generate ribosome


../scripts/expression/protein_expression/ubiquitin.py:34 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../scripts/expression/protein_expression/ubiquitin.py:64 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../scripts/expression/gene_information.py:113 UserWarning: HGNC:10368: The letter X is in the protein sequence. Replacing with a random amino acid
  1%|          | 6/591 [00:00<00:10, 53.76it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:12<00:00, 45.78it/s]


...

A total of 1897 reactions were dropped when forming a minimal proteome
Add machinery to metabolic module reactions


  0%|          | 33/10649 [00:00<00:33, 320.58it/s]

Add machinery to expression module reactions


100%|██████████| 10649/10649 [00:43<00:00, 247.43it/s]


Generate ME-Model
Time to build: 4.384248999754588 minutes
Getting MINOS parameters...
Done in 158.844 seconds with status 0


In [6]:
import pandas as pd
res = pd.DataFrame(data = {'reaction_fluxes': sln[:len(toy_me_model.reactions)]})
res.index = [r.id for r in toy_me_model.reactions]

In [7]:
res.loc[[i for i in res.index if 'biomass' in i],:]

,reaction_fluxes
biomass_dilution,1.000000e-09
DNA_biomass_to_biomass,1.400000e-11
carbohydrate_biomass_to_biomass,7.100000e-11
lipid_biomass_to_biomass,9.700000e-11
tRNA_biomass_to_biomass,1.132368e-22
rRNA_biomass_to_biomass,2.305160e-14
mRNA_biomass_to_biomass,3.701969e-10
premRNA_biomass_to_biomass,-8.292069e-22
other_rna_biomass_to_biomass,5.452825e-22
DNA_biomass_formation,1.000000e-09


In [10]:
import pickle
lp_path = '/data2/hratch/human_me/test_lp/'
with open(lp_path + 'working_version_' + str(0) + '.pickle', 'wb') as handle:
    pickle.dump(toy_me_model, handle)

S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
S.to_hdf(fn, key = str(0), mode = 'w')

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '0'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
